# Summary Catalog

Cross-provider data coverage dashboard. One row per ticker in the `catalog` table,
rebuilt via `uv run irp` → Steps → `catalog`.

Aggregates coverage from `prices` (Stooq), `yahoo_prices`, `dividends`, `splits`,
`income`, `balance`, `cashflow`, `companies`, and Yahoo fetch JSON files.

In [1]:
import pandas as pd
from IPython.display import display

from irp.data._common import db
from irp.data.catalog import catalog

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## Table

Schema, row count, and key stats for the `catalog` table.
Rebuilt on demand — does not update automatically when sources are stored.

### `catalog`

One row per ticker (base: `DISTINCT Ticker, Market FROM markets`).

| Column | Type | Source |
|---|---|---|
| Ticker | VARCHAR | markets |
| Market | VARCHAR | markets |
| stooq_first / stooq_last | DATE | prices |
| stooq_rows | BIGINT | prices |
| yahoo_first / yahoo_last | DATE | yahoo_prices |
| yahoo_rows | BIGINT | yahoo_prices |
| yahoo_prices_queried | BOOLEAN | queried_prices.json |
| yahoo_prices_error | BOOLEAN | error_tickers.json |
| yahoo_actions_queried | BOOLEAN | queried_actions.json |
| yahoo_actions_error | BOOLEAN | error_tickers.json |
| div_count | BIGINT | dividends |
| div_first / div_last | DATE | dividends |
| split_count | BIGINT | splits |
| income_A / income_Q | BIGINT | income |
| income_latest_yr | BIGINT | income |
| balance_A / balance_Q | BIGINT | balance |
| cashflow_A / cashflow_Q | BIGINT | cashflow |
| in_companies | BOOLEAN | companies |
| catalog_updated_at | TIMESTAMP | — |

In [2]:
_sample = catalog('AAPL')
display(_sample.dtypes.to_frame('dtype'))
display(_sample.T)

,dtype
Ticker,str
Market,str
stooq_first,datetime64[us]
stooq_last,datetime64[us]
stooq_rows,int64
yahoo_first,datetime64[us]
yahoo_last,datetime64[us]
yahoo_rows,int64
yahoo_prices_queried,bool
yahoo_prices_error,bool


,0
Ticker,AAPL
Market,nasdaq stocks
stooq_first,1984-09-07 00:00:00
stooq_last,2026-05-15 00:00:00
stooq_rows,10503
yahoo_first,1980-12-12 00:00:00
yahoo_last,2026-05-18 00:00:00
yahoo_rows,11449
yahoo_prices_queried,True
yahoo_prices_error,False


In [3]:
_stats = db().execute("""
    SELECT
        COUNT(*)                          AS tickers,
        COUNT(DISTINCT Market)            AS markets,
        SUM(stooq_rows > 0)              AS have_stooq,
        SUM(yahoo_rows > 0)              AS have_yahoo,
        SUM(div_count > 0)               AS have_dividends,
        SUM(split_count > 0)             AS have_splits,
        SUM(income_A > 0)                AS have_income,
        SUM(in_companies)                AS have_company_meta,
        MAX(catalog_updated_at)          AS last_rebuilt
    FROM catalog
""").df().T
_stats.columns = ['catalog']
display(_stats)

,catalog
tickers,14532
markets,11
have_stooq,14532.0
have_yahoo,11576.0
have_dividends,6979.0
have_splits,3274.0
have_income,2981.0
have_company_meta,4138.0
last_rebuilt,2026-05-18 21:14:11.021123+02:00


## Coverage by Market

For each market: how many tickers have data in each source.

In [4]:
display(db().execute("""
    SELECT
        Market,
        COUNT(*)                AS tickers,
        SUM(stooq_rows > 0)    AS stooq,
        SUM(yahoo_rows > 0)    AS yahoo,
        SUM(div_count > 0)     AS dividends,
        SUM(split_count > 0)   AS splits,
        SUM(income_A > 0)      AS income,
        SUM(balance_A > 0)     AS balance,
        SUM(cashflow_A > 0)    AS cashflow,
        SUM(in_companies)      AS companies
    FROM catalog
    GROUP BY Market
    ORDER BY tickers DESC
""").df())

,Market,tickers,stooq,yahoo,dividends,splits,income,balance,cashflow,companies
0,nasdaq stocks,4643,4643.0,4468.0,1290.0,1630.0,1622.0,1622.0,1622.0,2271.0
1,nyse stocks,3670,3670.0,3269.0,2474.0,980.0,1228.0,1228.0,1228.0,1664.0
2,nyse etfs,2552,2552.0,2552.0,2235.0,415.0,5.0,5.0,5.0,13.0
3,currencies,1822,1822.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,nasdaq etfs,938,938.0,937.0,843.0,103.0,1.0,1.0,1.0,3.0
5,nysemkt stocks,306,306.0,283.0,105.0,125.0,90.0,90.0,90.0,143.0
6,bonds,284,284.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,cryptocurrencies,215,215.0,52.0,32.0,21.0,35.0,35.0,35.0,44.0
8,indices,62,62.0,15.0,0.0,0.0,0.0,0.0,0.0,0.0
9,money market,27,27.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Yahoo Fetch Status

Download progress per market: queried, errors, confirmed rows, and completion rate.

In [5]:
display(db().execute("""
    SELECT
        Market,
        COUNT(*)                                                        AS target,
        SUM(yahoo_prices_queried)                                      AS prices_queried,
        SUM(yahoo_prices_error)                                        AS prices_errors,
        SUM(yahoo_rows > 0)                                            AS has_prices,
        SUM(yahoo_actions_queried)                                     AS actions_queried,
        SUM(yahoo_actions_error)                                       AS actions_errors,
        ROUND(SUM(yahoo_rows > 0) * 100.0 / COUNT(*), 1)              AS pct_downloaded
    FROM catalog
    GROUP BY Market
    ORDER BY target DESC
""").df())

,Market,target,prices_queried,prices_errors,has_prices,actions_queried,actions_errors,pct_downloaded
0,nasdaq stocks,4643,4334.0,309.0,4468.0,11.0,309.0,96.2
1,nyse stocks,3670,3181.0,489.0,3269.0,3.0,489.0,89.1
2,nyse etfs,2552,2460.0,92.0,2552.0,2.0,92.0,100.0
3,currencies,1822,0.0,1822.0,0.0,0.0,1822.0,0.0
4,nasdaq etfs,938,894.0,44.0,937.0,2.0,44.0,99.9
5,nysemkt stocks,306,275.0,31.0,283.0,0.0,31.0,92.5
6,bonds,284,0.0,0.0,0.0,0.0,0.0,0.0
7,cryptocurrencies,215,52.0,0.0,52.0,0.0,0.0,24.2
8,indices,62,14.0,48.0,15.0,0.0,48.0,24.2
9,money market,27,0.0,0.0,0.0,0.0,0.0,0.0


## Cross-provider Gaps

Tickers present in one price source but not the other.

In [6]:
display(db().execute("""
    SELECT
        Market,
        SUM(stooq_rows > 0 AND yahoo_rows > 0)  AS both,
        SUM(stooq_rows > 0 AND yahoo_rows = 0)  AS stooq_only,
        SUM(yahoo_rows > 0 AND stooq_rows = 0)  AS yahoo_only,
        SUM(stooq_rows = 0 AND yahoo_rows = 0)  AS neither
    FROM catalog
    GROUP BY Market
    ORDER BY stooq_only + yahoo_only DESC
""").df())

,Market,both,stooq_only,yahoo_only,neither
0,currencies,0.0,1822.0,0.0,0.0
1,nyse stocks,3269.0,401.0,0.0,0.0
2,bonds,0.0,284.0,0.0,0.0
3,nasdaq stocks,4468.0,175.0,0.0,0.0
4,cryptocurrencies,52.0,163.0,0.0,0.0
5,indices,15.0,47.0,0.0,0.0
6,money market,0.0,27.0,0.0,0.0
7,nysemkt stocks,283.0,23.0,0.0,0.0
8,stooq stocks indices,0.0,13.0,0.0,0.0
9,nasdaq etfs,937.0,1.0,0.0,0.0


## Freshness

Latest stored date per source, per market.

In [7]:
display(db().execute("""
    SELECT
        Market,
        MAX(stooq_last)   AS stooq_latest,
        MAX(yahoo_last)   AS yahoo_latest,
        MAX(div_last)     AS div_latest
    FROM catalog
    GROUP BY Market
    ORDER BY Market
""").df())

,Market,stooq_latest,yahoo_latest,div_latest
0,bonds,2026-05-15,NaT,NaT
1,cryptocurrencies,2026-05-16,2026-05-18,2026-05-15
2,currencies,2026-05-15,NaT,NaT
3,indices,2026-05-15,2026-05-18,NaT
4,money market,2026-05-15,NaT,NaT
5,nasdaq etfs,2026-05-16,2026-05-18,2026-05-15
6,nasdaq stocks,2026-05-16,2026-05-18,2026-05-15
7,nyse etfs,2026-05-16,2026-05-18,2026-05-15
8,nyse stocks,2026-05-16,2026-05-18,2026-05-15
9,nysemkt stocks,2026-05-16,2026-05-18,2026-05-15


## History Depth

Average years of price history per market. Tickers with 10+ years vs under 1 year.

In [8]:
display(db().execute("""
    SELECT
        Market,
        COUNT(*) FILTER (WHERE stooq_rows > 0)                                                              AS stooq_tickers,
        ROUND(AVG(DATEDIFF('day', stooq_first, CURRENT_DATE) / 365.0) FILTER (WHERE stooq_rows > 0), 1)    AS stooq_avg_yrs,
        COUNT(*) FILTER (WHERE yahoo_rows > 0)                                                              AS yahoo_tickers,
        ROUND(AVG(DATEDIFF('day', yahoo_first, CURRENT_DATE) / 365.0) FILTER (WHERE yahoo_rows > 0), 1)    AS yahoo_avg_yrs,
        COUNT(*) FILTER (WHERE yahoo_rows > 0 AND yahoo_first <= CURRENT_DATE - INTERVAL '10 years')        AS yahoo_ge_10yr,
        COUNT(*) FILTER (WHERE yahoo_rows > 0 AND yahoo_first > CURRENT_DATE - INTERVAL '1 year')           AS yahoo_lt_1yr
    FROM catalog
    GROUP BY Market
    ORDER BY stooq_avg_yrs DESC NULLS LAST
""").df())  

,Market,stooq_tickers,stooq_avg_yrs,yahoo_tickers,yahoo_avg_yrs,yahoo_ge_10yr,yahoo_lt_1yr
0,indices,62,44.7,15,32.0,13,0
1,currencies,1822,35.8,0,NaN,0,0
2,money market,27,20.4,0,NaN,0,0
3,bonds,284,19.0,0,NaN,0,0
4,stooq stocks indices,13,12.0,0,NaN,0,0
5,nyse stocks,3670,11.6,3269,16.3,1625,479
6,nysemkt stocks,306,11.5,283,17.9,173,15
7,nyse etfs,2552,9.3,2552,9.6,959,1
8,nasdaq stocks,4643,8.0,4468,11.0,1553,1065
9,cryptocurrencies,215,8.0,52,18.5,33,0


## SimFin Depth

Fundamental coverage: how many annual and quarterly periods stored per market.

In [9]:
display(db().execute("""
    SELECT
        Market,
        SUM(income_A > 0)                                               AS have_income,
        ROUND(AVG(income_A)  FILTER (WHERE income_A > 0),  1)          AS avg_annual_periods,
        ROUND(AVG(income_Q)  FILTER (WHERE income_Q > 0),  1)          AS avg_qtr_periods,
        MAX(income_latest_yr)                                           AS latest_yr,
        SUM(in_companies)                                               AS have_company_meta
    FROM catalog
    GROUP BY Market
    ORDER BY have_income DESC
""").df())

,Market,have_income,avg_annual_periods,avg_qtr_periods,latest_yr,have_company_meta
0,nasdaq stocks,1622.0,4.3,15.2,2025,2271.0
1,nyse stocks,1228.0,4.2,15.6,2025,1664.0
2,nysemkt stocks,90.0,4.0,12.4,2025,143.0
3,cryptocurrencies,35.0,4.1,13.8,2025,44.0
4,nyse etfs,5.0,2.2,9.4,2024,13.0
5,nasdaq etfs,1.0,2.0,7.0,2022,3.0
6,indices,0.0,NaN,NaN,<NA>,0.0
7,currencies,0.0,NaN,NaN,<NA>,0.0
8,bonds,0.0,NaN,NaN,<NA>,0.0
9,stooq stocks indices,0.0,NaN,NaN,<NA>,0.0
